In [1]:
import os
import re
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model

2025-12-20 00:06:18.761826: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766189178.989253      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766189179.052960      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766189179.602725      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766189179.602786      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766189179.602789      55 computation_placer.cc:177] computation placer alr

In [2]:
TEST_DIR = "/kaggle/input/test-image-script"
WEIGHTS_PATH = "/kaggle/input/weights-folder/best_efficientnetv2b0.weights.h5"
CLASS_INDICES_PATH = "/kaggle/input/class-index/class_indices.json"
CALORIES_FILE = "/kaggle/input/calories-folders/Calories.txt"

IMAGE_SIZE = (224, 224)

In [3]:
with open(CLASS_INDICES_PATH, "r") as f:
    class_indices = json.load(f)

idx_to_class = {v: k for k, v in class_indices.items()}
NUM_CLASSES = len(idx_to_class)
print("✅ Loaded class indices:", idx_to_class)

✅ Loaded class indices: {0: 'Apple_Gala', 1: 'Apple_Golden Delicious', 2: 'Avocado', 3: 'Banana', 4: 'Berry', 5: 'Burmese Grape', 6: 'Carambola', 7: 'Date Palm', 8: 'Dragon', 9: 'Elephant Apple', 10: 'Grape', 11: 'Green Coconut', 12: 'Guava', 13: 'Hog Plum', 14: 'Kiwi', 15: 'Lichi', 16: 'Malta', 17: 'Mango Golden Queen', 18: 'Mango_Alphonso', 19: 'Mango_Amrapali', 20: 'Mango_Bari', 21: 'Mango_Himsagar', 22: 'Olive', 23: 'Orange', 24: 'Palm', 25: 'Persimmon', 26: 'Pineapple', 27: 'Pomegranate', 28: 'Watermelon', 29: 'White Pear'}


In [4]:
def load_calories(file_path):
    calories = {}
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = re.match(r"(.+): ~([\d\.]+) calories per gram", line)
            if match:
                food = match.group(1).strip()
                cal = float(match.group(2))
                calories[food] = cal
    return calories

calories_dict = load_calories(CALORIES_FILE)
print("✅ Calories file loaded:", calories_dict)

✅ Calories file loaded: {'Pomegranate': 0.83, 'Watermelon': 0.3, 'White Pear': 0.57, 'Palm': 2.8, 'Persimmon': 0.81, 'Pineapple': 0.5, 'Olive': 1.15, 'Orange': 0.47, 'Mango_Amrapali': 0.65, 'Mango_Bari': 0.65, 'Mango_Himsagar': 0.65, 'Lichi': 0.66, 'Malta': 0.47, 'Mango Golden Queen': 0.65, 'Mango_Alphonso': 0.65, 'Hog Plum': 0.5, 'Kiwi': 0.61, 'Green Coconut': 1.9, 'Guava': 0.68, 'Elephant Apple': 0.6, 'Grape': 0.69, 'Date Palm': 2.8, 'Dragon': 0.6, 'Carambola': 0.31, 'Berry': 0.5, 'Burmese Grape': 0.55, 'Avocado': 1.6, 'Banana': 0.89, 'Apple_Gala': 0.52, 'Apple_Golden Delicious': 0.52}


In [5]:
base_model = EfficientNetV2B0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.load_weights(WEIGHTS_PATH)
print("✅ Model weights loaded successfully")

2025-12-20 00:06:33.864074: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


24274472/24274472 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
✅ Model weights loaded successfully


In [6]:
def extract_grams(filename):
    match = re.search(r"_(\d+)g", filename)
    if match:
        return int(match.group(1))
    return None


In [7]:
output_file = "calorie_predictions.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("🔍 CALORIE PREDICTIONS\n\n")

    for root, _, files in os.walk(TEST_DIR):
        for img_name in files:
            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            grams = extract_grams(img_name)
            if grams is None:
                line = f"⚠️ Skipped (no grams): {img_name}\n"
                f.write(line)
                continue

            img_path = os.path.join(root, img_name)
            img = load_img(img_path, target_size=IMAGE_SIZE)
            img = img_to_array(img)
            img = np.expand_dims(img, axis=0)
            img = preprocess_input(img)

            preds = model.predict(img, verbose=0)
            pred_idx = np.argmax(preds)
            confidence = float(np.max(preds))
            pred_class = idx_to_class[pred_idx]

            if pred_class not in calories_dict:
                line = f"⚠️ Calories not found for {pred_class}\n"
                f.write(line)
                continue

            total_calories = grams * calories_dict[pred_class]

            line = (
                f"🖼️ {img_name} | "
                f"Predicted: {pred_class} | "
                f"Confidence: {confidence:.2f} | "
                f"Weight: {grams}g | "
                f"Calories: {total_calories:.2f} kcal\n"
            )
            f.write(line)

print(f"✅ All predictions saved to '{output_file}'")

✅ All predictions saved to 'calorie_predictions.txt'
